# OpenMotion AI - Backend com LTX-Video (Versão Corrigida 2026)
Mais rápida e estável para 720p + motion control (dança)

In [ ]:
!nvidia-smi

import os
import shutil
os.chdir('/content')
if os.path.exists('ltx_backend'):
    shutil.rmtree('ltx_backend')

!git clone https://github.com/Lightricks/LTX-Video.git ltx_backend
os.chdir('/content/ltx_backend')

# Instalações corrigidas para T4
!pip install -U diffusers transformers accelerate xformers torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install mediapipe opencv-python pillow fastapi uvicorn python-multipart pyngrok

import torch
torch.backends.cudnn.benchmark = True
print("✅ Ambiente pronto")

### Criar arquivos do backend (corrigido)

In [ ]:
%%writefile /content/ltx_backend/backend/main.py
import os
import torch
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import FileResponse
import uuid
import shutil
from pathlib import Path
from pipeline_ltx import LTXPosePipeline

app = FastAPI(title="OpenMotion AI - LTX Backend")

# Carrega uma única vez
print("Carregando LTX-Video (pode demorar 30-60s)...")
global_pipeline = LTXPosePipeline()
print("✅ Pipeline carregado")

@app.post("/generate")
async def generate(
    character_image: UploadFile = File(...),
    reference_video: UploadFile = File(...),
    prompt: str = Form("a person dancing, high quality, detailed face, realistic"),
    negative_prompt: str = Form("blurry, low quality, deformed, ugly, bad anatomy")
):
    task_id = str(uuid.uuid4())[:8]
    temp_dir = Path(f"/content/temp_{task_id}")
    temp_dir.mkdir(exist_ok=True)

    try:
        char_path = temp_dir / "character.png"
        vid_path = temp_dir / "reference.mp4"

        with open(char_path, "wb") as f: f.write(await character_image.read())
        with open(vid_path, "wb") as f: f.write(await reference_video.read())

        output_path = await global_pipeline.generate(
            character_path=str(char_path),
            reference_video=str(vid_path),
            prompt=prompt,
            negative_prompt=negative_prompt,
            output_dir=str(temp_dir)
        )

        return FileResponse(output_path, media_type="video/mp4", filename=f"dance_{task_id}.mp4")

    finally:
        if temp_dir.exists():
            shutil.rmtree(temp_dir, ignore_errors=True)

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)

In [ ]:
%%writefile /content/ltx_backend/backend/pipeline_ltx.py
import torch
from diffusers import LTXPipeline, LTXImageToVideoPipeline   # pipelines corretos
from PIL import Image
import cv2
from pathlib import Path

class LTXPosePipeline:
    def __init__(self):
        self.device = "cuda"
        # Usamos a pipeline Image-to-Video (mais estável no Diffusers)
        self.pipe = LTXImageToVideoPipeline.from_pretrained(
            "Lightricks/LTX-Video",
            torch_dtype=torch.bfloat16,
            variant="fp16"
        ).to(self.device)

        self.pipe.enable_xformers_memory_efficient_attention()
        self.pipe.enable_vae_slicing()

    async def generate(self, character_path, reference_video, prompt, negative_prompt, output_dir):
        # Carrega imagem da pessoa
        character_img = Image.open(character_path).convert("RGB")
        character_img = character_img.resize((1280, 720))  # força 720p

        # Extrai alguns frames do vídeo de referência (para conditioning)
        cap = cv2.VideoCapture(reference_video)
        frames = []
        for i in range(8):  # pega ~8 frames iniciais como referência de movimento
            ret, frame = cap.read()
            if not ret:
                break
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(rgb).resize((1280, 720)))
        cap.release()

        # Geração principal
        output = self.pipe(
            image=character_img,                    # nova pessoa
            prompt=prompt,
            negative_prompt=negative_prompt,
            num_inference_steps=20,
            guidance_scale=6.5,
            height=720,
            width=1280,
            num_frames=48,                          # ~2 segundos a 24fps (aumente se quiser mais longo)
            generator=torch.Generator(device="cuda").manual_seed(42),
        )

        video_path = Path(output_dir) / "output.mp4"

        # Exporta vídeo corretamente
        from diffusers.utils import export_to_video
        export_to_video(output.frames[0], str(video_path), fps=24)

        return str(video_path)